# レッスン03 - エージェント設計パターン

このレッスンでは、効果的なAIエージェントを構築するための3つの基本的な設計パターンを探ります：

1. <strong>明確なエージェント指示</strong> — エージェントの行動を導く、正確で役割を定義したプロンプトの作成
2. **Pydanticモデルによる構造化された出力** — 予測可能で検証済みのデータを返すエージェントの保証
3. <strong>単一責任エージェント</strong> — それぞれが1つのことをうまく行う集中型エージェントの設計

これらのパターンを<strong>旅行先推薦システム</strong>のシナリオに適用し、目的地の提案、空き状況の確認、ロジスティクスの処理ができるシステムを段階的に構築していきます。


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## パターン 1: 明確なエージェント指示

最も効果的なパターンは最もシンプルなものでもあります：エージェントに対して明確で詳細な指示を書くことです。

良い指示は以下を定義します：
- <strong>誰が</strong> エージェントであるか（ペルソナとトーン）
- <strong>何を</strong> すべきか（段階的な責任）
- <strong>どのように</strong> 振る舞うべきか（制約とスタイル）

以下では、すべての応答に形を与える明示的な指示を持つ旅行コンシェルジュエージェントを作成します。


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## パターン2: Pydanticモデルによる構造化された出力

自由形式のテキストは会話には便利ですが、下流のシステムは構造化されたデータを必要とします。
<strong>Pydanticモデル</strong>と<strong>ツール関数</strong>を組み合わせることで、以下が可能になります:

- エージェントの出力に対して正確なスキーマを定義
- 応答の自動検証
- エージェントの結果を確実にアプリケーションロジックに統合

この強制の鍵は、エージェントを実行する際に`response_format`を渡すことです。これにより、
モデルは自由形式のテキストではなく、検証済みの`TravelRecommendations`オブジェクト（`response.value`で利用可能）を返します。
`get_destination_details`ツールも型付きの`DestinationRecommendation`を返すため、
データは端から端まで構造化されたままです。


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## パターン 3: 単一責任エージェント

複雑なタスクは、単一の責任を持つ複数の専門化されたエージェントに作業を分割することで効果的になります：

- 場所や空き情報に詳しい <strong>目的地の専門家</strong>
- フライト、ホテル、旅程を担当する <strong>ロジスティクスプランナー</strong>

これはソフトウェア工学の原則である <em>関心の分離</em> に似ており — 各エージェントは独立してテスト、保守、改善がしやすくなります。


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## まとめ

このレッスンでは、旅行推薦シナリオに三つのエージェント設計パターンを適用しました：

| パターン | 主要アイデア | 利点 |
|---|---|---|
| <strong>明確な指示</strong> | ペルソナ、責任、制約を事前に定義する | 一貫したブランドに沿ったエージェントの振る舞い |
| <strong>構造化された出力</strong> | Pydanticモデルを応答形式として使用する | 検証済みで機械可読な結果 |
| <strong>単一責任</strong> | 各エージェントに1つの集中した仕事を与える | テスト、保守、組み合わせが容易になる |

これらのパターンは自然に組み合わさり、単一責任のエージェント内で明確な指示と構造化された出力を組み合わせて堅牢で本番対応可能なシステムを構築できます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
